# Hybrid RAG with Pinecone having semantic & syntactic search

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
pinecone_api_key=os.getenv('PINECONE_API_KEY')

In [3]:
from langchain_community.retrievers import PineconeHybridSearchRetriever # this class can do both syntactic (dense vec) & semantic(sparse) search

In [4]:
from pinecone import Pinecone, ServerlessSpec   # automatically detect if we are using serverless vec DB or not and use the appropriate client
import pinecone as pinecone_module
# Monkeypatch Index.upsert to support positional arguments used by older langchain-community versions
old_upsert = pinecone_module.Index.upsert
def patched_upsert(self, *args, **kwargs):
    if len(args) > 0:
        kwargs['vectors'] = args[0]
        args = args[1:]
    return old_upsert(self, *args, **kwargs)
pinecone_module.Index.upsert = patched_upsert

index_name = "hybrid-search-langchain-pinecone"

## Initialize the Pinecone client
pinecone = Pinecone(api_key=pinecone_api_key)

In [6]:
## create index 
if index_name not in [idx.name for idx in pinecone.list_indexes()]:   ## check if the index already exists or not to avoid error, if not exists then create it
    pinecone.create_index(
        name=index_name,
        dimension=384,              # dimension of the dense vectors - for HF embeddings it is 384
        metric="dotproduct",        # similarity metric to use for sparse vector search (and for dense vector search)
        spec=ServerlessSpec(cloud="aws", region="us-east-1")    # use serverless index for simplicity, you can also use provisioned index if you want more control over the resources and performance
    )

In [7]:
index = pinecone.Index(index_name)
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=384, total_vector_count=0, metric='dotproduct', namespaces=0)

In [8]:
## Hybrid search = vec embedding for dense search (syntactic) + sparse search (semantic)

## for dense search
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv('HUGGINGFACEHUB_API_TOKEN')

from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings  ## used for dense vectors

c:\All_Work\LangChainProj\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4754.09it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [9]:
## for sparse search

from pinecone_text.sparse import BM25Encoder # a sparse encoder that uses BM25 algorithm (TF-IDF) to encode the text into sparse vectors

bm25_encoder = BM25Encoder().default()  ## initialize the BM25 encoder with default param
bm25_encoder  ## used for sparse vectors

In [21]:
my_text = [
    "The cat is on the table.",
    "The dog is in the garden.",
    "The bird is flying in the sky.",
    "The cow is grazing in the field.",
    "The fish is swimming in the pond."
]

In [ ]:
## tfidf sparse encoding
bm25_encoder.fit(my_text)

## store the values to a json file to see the sparse vectors
import json
bm25_encoder.dump("bm25_values.json")

## load BM25Encoder object
bm25_encoder_loaded = BM25Encoder().load("bm25_values.json")

# So, instead of recomputing the same statis every time we run the notebook, we can load the saved JSON and reuse the exact same encoder.

100%|██████████| 5/5 [00:00<00:00, 9887.56it/s]


In [ ]:
## support both dense and sparse search with PineconeHybridSearchRetriever
retriever = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25_encoder_loaded, index=index, top_k=5)
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x000001FE4C70F8B0>, index=Index(host='https://hybrid-search-langchain-pinecone-e0sh35d.svc.aped-4627-b74a.pinecone.io'), top_k=3)

In [ ]:
retriever.add_texts(my_text)
# gets inserted in  the pinecone index as a single vector that combines both dense and sparse vectors, and the retriever can use both vectors to perform hybrid search.

100%|██████████| 1/1 [00:00<00:00,  1.32it/s]


In [26]:
retriever.invoke("Where is the dog?")  ## should return the first text as the most relevant result with high score, and also return the other texts with lower scores based on the sparse search results.

[Document(metadata={'score': 0.526786566}, page_content='The dog is in the garden.'),
 Document(metadata={'score': 0.486271977}, page_content='The dog is in the garden.'),
 Document(metadata={'score': 0.135585785}, page_content='The cat is on the table.')]